Note: this notebook should be run with the r-seurat-h5 kernel

In [1]:
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(rhdf5)
    library(arrow)
})

In [8]:
allcells <- readRDS('out_rds/allcells_qc_harm_umap_clusters_annotation_tessera.rds')

cols_to_drop <- c("humap_fgraph_res.0.8", "seurat_clusters")

allcells@meta.data <- allcells@meta.data[, 
  !colnames(allcells@meta.data) %in% cols_to_drop, 
  drop = FALSE
]

In [8]:
i <- 1

In [9]:
dir.create("level4/", showWarnings = FALSE, recursive = TRUE)

counts_mat <- Seurat::GetAssayData(allcells, layer = "counts")
meta <- allcells@meta.data

sids <- unique(meta$sid)
for (sid in sids[i:length(sids)]) {
  message("Processing ", sid)

  cells_use <- rownames(meta)[meta$sid == sid]

  # subset counts and metadata for this sample
  sub_counts <- counts_mat[, cells_use, drop = FALSE]
  sub_meta <- meta[cells_use, , drop = FALSE]

  h5_file <- file.path("level4", paste0(sid, ".h5"))
  pq_file <- file.path("level4", paste0(sid, ".parquet"))
  if (file.exists(h5_file)) file.remove(h5_file)
  if (file.exists(pq_file)) file.remove(pq_file)

  # Store three separate objects in the h5 file
  # counts is written as a regular matrix here
  rhdf5::h5createFile(h5_file)
  rhdf5::h5write(as.matrix(sub_counts), h5_file, "counts")
  rhdf5::h5write(colnames(sub_counts), h5_file, "cells")
  rhdf5::h5write(rownames(sub_counts), h5_file, "genes")

  # Write metadata to parquet, preserving cell ids as a column
  sub_meta_out <- cbind(cell_id = rownames(sub_meta), sub_meta)
  arrow::write_parquet(sub_meta_out, pq_file)
  message("Finished sample #", i)
  i <- i+1
}

Processing 8073578341-02-02

You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the dataset, you should testsmaller chunk sizes to improve read times.

Finished sample #1

Processing 8073579095-02-02

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 6.0 GiB”
Current chunk settings will exceed HDF5's 4GB limit.
Automatically adjusting them to: 4008 x 23170
You may wish to set these to more appropriate values using the 'chunk' argument.

Finished sample #2

Processing 8073578234-02-02

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.2 GiB”
You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the dataset, you should testsmaller chunk sizes to improve read times.

Finished sample #3

Processing 8073578941-02-02

Warning message in asMe

# spot check

In [12]:
sid <- "8073578234-02-02"

h5_file <- file.path("level4", paste0(sid, ".h5"))
pq_file <- file.path("level4", paste0(sid, ".parquet"))

stopifnot(file.exists(h5_file))
stopifnot(file.exists(pq_file))

# ---------------------------
# Read data back in
# ---------------------------
counts <- rhdf5::h5read(h5_file, "counts")
cells  <- rhdf5::h5read(h5_file, "cells")
genes  <- rhdf5::h5read(h5_file, "genes")
meta   <- arrow::read_parquet(pq_file)

# convert parquet result to plain data.frame if needed
meta <- as.data.frame(meta)

# attach dimnames to counts
stopifnot(length(dim(counts)) == 2)
stopifnot(nrow(counts) == length(genes))
stopifnot(ncol(counts) == length(cells))

rownames(counts) <- genes
colnames(counts) <- cells

# ---------------------------
# Basic completeness checks
# ---------------------------
cat("=== File existence ===\n")
cat("H5 exists:      ", file.exists(h5_file), "\n")
cat("Parquet exists: ", file.exists(pq_file), "\n\n")

cat("=== Dimensions ===\n")
cat("n_genes in H5:  ", nrow(counts), "\n")
cat("n_cells in H5:  ", ncol(counts), "\n")
cat("rows in meta:   ", nrow(meta), "\n\n")

cat("=== Required metadata columns ===\n")
cat("Has cell_id column: ", "cell_id" %in% colnames(meta), "\n")
cat("Has sid column:     ", "sid" %in% colnames(meta), "\n\n")

cat("=== Missingness / uniqueness ===\n")
cat("Any NA in genes:         ", any(is.na(genes)), "\n")
cat("Any NA in cells:         ", any(is.na(cells)), "\n")
cat("Duplicated genes:        ", anyDuplicated(genes), "\n")
cat("Duplicated cells:        ", anyDuplicated(cells), "\n")
cat("Duplicated meta cell_id: ",
    if ("cell_id" %in% colnames(meta)) anyDuplicated(meta$cell_id) else NA, "\n\n")

# ---------------------------
# Cross-check metadata vs H5
# ---------------------------
if (!("cell_id" %in% colnames(meta))) {
  stop("Metadata is missing required column: cell_id")
}

cells_in_h5_not_meta   <- setdiff(cells, meta$cell_id)
cells_in_meta_not_h5   <- setdiff(meta$cell_id, cells)
order_matches          <- all(cells == meta$cell_id)

cat("=== Cell ID agreement ===\n")
cat("Cells in H5 but not meta: ", length(cells_in_h5_not_meta), "\n")
cat("Cells in meta but not H5: ", length(cells_in_meta_not_h5), "\n")
cat("Cell order identical:     ", order_matches, "\n\n")

if (length(cells_in_h5_not_meta) > 0) {
  cat("First few cells in H5 not meta:\n")
  print(head(cells_in_h5_not_meta))
  cat("\n")
}

if (length(cells_in_meta_not_h5) > 0) {
  cat("First few cells in meta not H5:\n")
  print(head(cells_in_meta_not_h5))
  cat("\n")
}

# ---------------------------
# Check sid consistency
# ---------------------------
if ("sid" %in% colnames(meta)) {
  unique_sids <- unique(meta$sid)
  cat("=== sid consistency ===\n")
  cat("Unique sid values in parquet:\n")
  print(unique_sids)
  cat("All sid values match requested sid: ", all(meta$sid == sid), "\n\n")
}

# ---------------------------
# Check for empty / suspicious data
# ---------------------------
cat("=== Matrix content checks ===\n")
cat("Storage mode:            ", typeof(counts), "\n")
cat("Any NA in counts:        ", any(is.na(counts)), "\n")
cat("All-zero columns:        ", sum(colSums(counts) == 0), "\n")
cat("All-zero rows:           ", sum(rowSums(counts) == 0), "\n")
cat("Total counts:            ", sum(counts), "\n")
cat("Nonzero entries:         ", sum(counts != 0), "\n\n")

# ---------------------------
# Final pass/fail summary
# ---------------------------
ok <- TRUE

ok <- ok && nrow(counts) == length(genes)
ok <- ok && ncol(counts) == length(cells)
ok <- ok && !any(is.na(genes))
ok <- ok && !any(is.na(cells))
ok <- ok && anyDuplicated(genes) == 0
ok <- ok && anyDuplicated(cells) == 0
ok <- ok && anyDuplicated(meta$cell_id) == 0
ok <- ok && length(cells_in_h5_not_meta) == 0
ok <- ok && length(cells_in_meta_not_h5) == 0

if ("sid" %in% colnames(meta)) {
  ok <- ok && all(meta$sid == sid)
}

cat("=== Overall result ===\n")
if (ok) {
  cat("PASS: sample file appears complete and internally consistent.\n")
} else {
  cat("FAIL: sample file has one or more consistency problems.\n")
}

=== File existence ===
H5 exists:       TRUE 
Parquet exists:  TRUE 

=== Dimensions ===
n_genes in H5:   4008 
n_cells in H5:   41822 
rows in meta:    41822 

=== Required metadata columns ===
Has cell_id column:  TRUE 
Has sid column:      TRUE 

=== Missingness / uniqueness ===
Any NA in genes:          FALSE 
Any NA in cells:          FALSE 
Duplicated genes:         0 
Duplicated cells:         0 
Duplicated meta cell_id:  0 

=== Cell ID agreement ===
Cells in H5 but not meta:  0 
Cells in meta but not H5:  0 
Cell order identical:      TRUE 

=== sid consistency ===
Unique sid values in parquet:
[1] "8073578234-02-02"
All sid values match requested sid:  TRUE 

=== Matrix content checks ===
Storage mode:             double 
Any NA in counts:         FALSE 
All-zero columns:         0 
All-zero rows:            0 
Total counts:             9314854 
Nonzero entries:          6023757 

=== Overall result ===
PASS: sample file appears complete and internally consistent.


In [13]:
meta

cell_id,orig.ident,nCount_RNA,nFeature_RNA,sid,x,y,avg_assignment_confidence,celltypes.coarse,tile_id,polygon_wkt
<chr>,<fct>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<chr>
CR41c523b5a-2_8073578234-02-02,EDP1,117,81,8073578234-02-02,4518.720,11324.95,0.9585,lining_fibroblast,8073578234-02-02_1,"POLYGON ((4521.328 11325.31, 4521.922 11323.72, 4521.828 11323.17, 4521.641 11322.84, 4521.328 11322.95, 4521.188 11322.75, 4518.094 11319.67, 4516.156 11321.77, 4515.531 11323.86, 4515.234 11325.09, 4515 11326.36, 4516.266 11328.2, 4517.125 11329.16, 4519.344 11328.55, 4520.047 11328.72, 4521.328 11325.31))"
CR41c523b5a-5_8073578234-02-02,EDP1,190,104,8073578234-02-02,4525.255,11327.41,0.9784,lining_fibroblast,8073578234-02-02_1,"POLYGON ((4526.188 11330.08, 4524.156 11330.05, 4523.078 11330.02, 4521.359 11329.02, 4521.734 11326.73, 4523.266 11321.81, 4525.484 11322.34, 4529.234 11322.95, 4526.016 11322.52, 4526.609 11324.22, 4528.781 11323.72, 4529.766 11325.88, 4529.531 11327.03, 4528.531 11329.33, 4526.703 11330, 4526.188 11330.08))"
CR41c523b5a-9_8073578234-02-02,EDP1,485,216,8073578234-02-02,4537.744,11325.16,0.9804,lining_fibroblast,8073578234-02-02_1,"POLYGON ((4535.688 11329.5, 4535.719 11331.11, 4539.281 11329.39, 4540.938 11330.66, 4540.922 11328.83, 4542.594 11328.22, 4543.875 11326.97, 4544.094 11326.47, 4544.141 11326.03, 4544.5 11322.2, 4540.469 11319.56, 4538.312 11319.42, 4531.125 11322.34, 4531.062 11325.67, 4531.797 11327.27, 4533.391 11327.02, 4535.688 11329.5))"
CR41c523b5a-11_8073578234-02-02,EDP1,126,95,8073578234-02-02,4532.660,11335.12,0.9519,sublining_fibroblast,8073578234-02-02_5,"POLYGON ((4530.625 11338.44, 4531.094 11338.7, 4534.297 11339.17, 4534.594 11337.27, 4536 11337.28, 4537.766 11332.97, 4537.219 11332.12, 4532.266 11332.03, 4531.203 11332.14, 4530.469 11330.73, 4527.562 11332.78, 4526.641 11335.33, 4530.625 11338.44))"
CR41c523b5a-17_8073578234-02-02,EDP1,318,172,8073578234-02-02,4548.669,11326.59,0.9608,lining_fibroblast,8073578234-02-02_5,"POLYGON ((4548.5 11332.58, 4551.406 11331.27, 4551.891 11330.47, 4554 11325.09, 4553.312 11324.62, 4550.719 11320.64, 4550.125 11320.69, 4545.516 11323.39, 4544.312 11324.23, 4544.188 11327.3, 4544.656 11329.09, 4545.391 11328.83, 4546.453 11330.8, 4547.422 11332.78, 4548.5 11332.58))"
CR41c523b5a-19_8073578234-02-02,EDP1,184,131,8073578234-02-02,4544.470,11339.75,0.9297,myeloid,8073578234-02-02_5,"POLYGON ((4543.281 11342.28, 4544.344 11342.38, 4545.25 11342.08, 4547.812 11337.89, 4547.422 11337.08, 4546.125 11336.8, 4543.875 11337.53, 4543.625 11337.64, 4543.078 11338.23, 4542.828 11338.62, 4542.141 11340.11, 4542.219 11341.5, 4542.359 11341.89, 4542.781 11342.23, 4543.062 11341.86, 4543.281 11342.28))"
CR41c523b5a-25_8073578234-02-02,EDP1,270,171,8073578234-02-02,4564.356,11326.98,0.9506,lining_fibroblast,8073578234-02-02_5,"POLYGON ((4570.797 11325.77, 4570.344 11327.17, 4570.672 11327.91, 4570.156 11328.64, 4569.422 11329.34, 4565.375 11331.47, 4560.625 11330.61, 4557.969 11326.88, 4557.812 11324.95, 4557.766 11324.19, 4558.016 11323.16, 4559.359 11322.66, 4560.109 11321.97, 4563.031 11322.55, 4565.25 11323.09, 4568.109 11324.36, 4570.797 11325.77))"
CR41c523b5a-26_8073578234-02-02,EDP1,183,108,8073578234-02-02,4567.733,11319.71,0.9740,myeloid,8073578234-02-02_5,"POLYGON ((4560.094 11320.44, 4560 11320.28, 4559.781 11317.72, 4560.125 11316.2, 4563.375 11315.64, 4566.438 11315.48, 4571.359 11316.69, 4573.766 11317.62, 4573.906 11321.67, 4572.984 11323.55, 4569.328 11324.02, 4564.406 11323.28, 4564.797 11321.81, 4562.75 11322.44, 4560.094 11320.44))"
CR41c523b5a-29_8073578234-02-02,EDP1,111,96,8073578234-02-02,4543.818,11343.90,0.8701,myeloid,8073578234-02-02_5,"POLYGON ((4543.109 11342.28, 4544.047 11342.72, 4544.594 11342.36, 4548.984 11343.7, 4546.734 11344.8, 4542.141 11345.47, 4541.469 11345.2, 4540.969 11344.94, 4541.938 11344.72, 4540.875 11344.72, 4541.016 11344.22, 4540.703 11344.08, 4543.078 11342.25, 4543.109 11342.28))"
